# 🎓 Projet 04 (Niveau L3) : "Initiation au Traitement de Données Solaire Photovoltaïque"

![Principe d'un panneau solaire](https://images.unsplash.com/photo-1509391365360-2e959784a276?w=900)

---

## 📖 COURS D'INTRODUCTION : Comment fonctionne une installation solaire ?

Bienvenue dans ce TP/Exercice guidé ! En tant qu'étudiant en  Énergie / Physique, tu vas apprendre à **analyser les mesures issues d'une centrale solaire photovoltaïque**.

### 1. Les grandeurs physiques mesurées sur le terrain
Pour suivre le bon fonctionnement d'une centrale solaire de surface $A_{champ} = 50 \text{ m}^2$, des capteurs enregistrent automatiquement à intervalles réguliers :
1. **$G$ (Irradiance Solaire en $\text{W/m}^2$)** : C'est la quantité d'énergie lumineuse reçue du Soleil par mètre carré. Plus il y a de soleil, plus $G$ est élevé.
2. **$T_{mod}$ (Température du panneau en $\text{°C}$)** : La température de la surface des panneaux. *Attention : quand un panneau chauffe trop, son rendement diminue !*
3. **$V$ (Tension en Volts)** et **$I$ (Courant en Ampères)** : Les grandeurs électriques produites par le panneau.

---

### 2. Les formules physiques fondamentales à retenir pour cet exercice
* **Puissance électrique produite ($P_{él}$ en kW)** :
  $$P_{él} = \frac{V \times I}{1000}$$

* **Puissance lumineuse reçue du Soleil ($P_{sol}$ en kW)** :
  $$P_{sol} = \frac{G \times A_{champ}}{1000}$$

* **Rendement de conversion du panneau ($\eta$ en %)** :
  $$\eta = \left( \frac{P_{él}}{P_{sol}} \right) \times 100$$

---

### 🎯 Objectifs de cet exercice pratique
Dans ce TP sous forme de code Python, tu vas apprendre pas à pas à :
1. Charger et inspecter un jeu de données de capteurs bruts.
2. Détecter et éliminer les "bugs" de mesure (valeurs aberrantes de température).
3. Calculer les nouvelles variables physiques (Puissance et Rendement).
4. Catégoriser la météo (`Faible`, `Moyen`, `Fort` ensoleillement).
5. Calculer le bilan énergétique global de l'installation.

## 🟢 Niveau 1 : Chargement & Découverte des Données Capteurs

### 📚 COURS : Représenter des mesures physiques avec Pandas
En Python, la bibliothèque `pandas` permet de manipuler des tableaux de données appelés des **DataFrames**. Chaque colonne représente une grandeur physique mesurée par un capteur ($G, T, V, I$), et chaque ligne représente un instant de mesure.

---

### ✏️ Exercice (Quête 1)
Des capteurs ont relevé 100 mesures sur notre champ solaire.
Convertis le dictionnaire de mesures fourni en un DataFrame Pandas `df_pv`, puis vérifie le nombre de mesures contenues.

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# QUÊTE 1 : Charger le tableau de relevés physiques
# ==========================================

# Dictionnaire simulant les mesures enregistrées par la centrale solaire
donnees_capteurs = {
    'irradiance_w_m2': [200.0, 400.0, 600.0, 800.0, 1000.0] * 20,
    'temp_module_c': [25.0, 30.0, 35.0, 40.0, 95.0] * 20,  # Remarque : 95°C est une mesure anormale (bug capteur) !
    'tension_v': [230.0, 228.0, 231.0, 229.0, 230.0] * 20,
    'courant_a': [5.0, 10.0, 15.0, 20.0, 25.0] * 20
}

# Consignes :
# 1. Transforme le dictionnaire `donnees_capteurs` en DataFrame Pandas nommé `df_pv`.
# 2. Compte le nombre total de lignes et stocke le résultat dans `nb_echantillons` (utilise len()).

# ### START CODE HERE ###

# ### END CODE HERE ###

# --- VERIFICATION AUTOMATIQUE ---
assert nb_echantillons == 100, f"Nombre d'échantillons attendu : 100, obtenu : {nb_echantillons}"
assert 'temp_module_c' in df_pv.columns, "La colonne 'temp_module_c' est manquante !"
print("✅ Bravo ! Quête 1 réussie : Les données des capteurs sont prêtes à être analysées.")

# --- AFFICHAGE DU GRAPHIQUE ---
from utils import tracer_inspection_energetique
tracer_inspection_energetique(df_pv)

## 🟡 Niveau 2 : Nettoyage des Données Aberrantes (Bugs Capteurs)

### 📚 COURS : Pourquoi et comment éliminer les fausses mesures ?
Sur le terrain, un capteur peut subir une micro-coupure ou un parasite électrique et enregistrer des données fausses (ex: un panneau affiché à **95°C** alors que la température réelle ne dépasse pas 40°C). Si on conserve ces erreurs, nos calculs de rendement seront faux !

Pour éliminer ces valeurs aberrantes de manière automatique, on utilise la méthode statistique de l'**Écart Interquartile (IQR)** :
1. **$Q_1$ (Premier Quartile)** : La valeur au-dessous de laquelle se trouvent 25% des mesures.
2. **$Q_3$ (Troisième Quartile)** : La valeur au-dessous de laquelle se trouvent 75% des mesures.
3. **$\text{IQR} = Q_3 - Q_1$** : L'écart entre ces deux quartiles.
4. **$\text{Seuil Haut} = Q_3 + 1.5 \times \text{IQR}$** : Toute température supérieure à ce seuil est considérée comme un bug et doit être éliminée.

---

### ✏️ Exercice (Quête 2)
Calcule les quartiles $Q_1$ et $Q_3$ de la température (`temp_module_c`), déduis-en le `seuil_sup`, puis conserve uniquement les mesures valides dans un nouveau DataFrame nommé `df_valide`.

In [ ]:
# ==========================================
# QUÊTE 2 : Nettoyer les fausses températures (Méthode IQR)
# ==========================================

# Consignes :
# 1. Calcule q1 (0.25) et q3 (0.75) pour la colonne 'temp_module_c' grâce à .quantile().
# 2. Calcule iqr = q3 - q1.
# 3. Calcule seuil_sup = q3 + 1.5 * iqr.
# 4. Filtre `df_pv` pour garder uniquement les lignes où 'temp_module_c' <= seuil_sup.
#    Stocke le résultat dans `df_valide`.

# ### START CODE HERE ###
q1 = df_pv['temp_module_c'].quantile(0.25)
q3 = df_pv['temp_module_c'].quantile(0.75)
iqr = q3 - q1
seuil_sup = q3 + 1.5 * iqr

df_valide = df_pv[df_pv['temp_module_c'] <= seuil_sup].copy()
# ### END CODE HERE ###

# --- VERIFICATION AUTOMATIQUE ---
assert len(df_valide) == 80, f"Nombre de lignes valides attendu : 80, obtenu : {len(df_valide)}"
assert df_valide['temp_module_c'].max() == 40.0, f"Température max attendue : 40°C, obtenue : {df_valide['temp_module_c'].max()}°C"
print("✅ Bravo ! Quête 2 réussie : Les 20 mesures aberrantes (95°C) ont été éliminées.")

# --- AFFICHAGE DU GRAPHIQUE ---
from utils import tracer_iqr_thermique
tracer_iqr_thermique(df_pv, df_valide, q1, q3, q1 - 1.5*iqr, seuil_sup)

## 🟠 Niveau 3 : Calculs Physiques Vectorisés & Régimes Météo

### 📚 COURS : Calculer de nouvelles grandeurs physiques dans un DataFrame
Grâce à Pandas, on peut réaliser des opérations sur des colonnes entières en une seule ligne (calcul vectorisé).

Dans cet exercice :
1. **La Puissance produite ($kW$)** se calcule par :
   $$\text{puissance\_kw} = \frac{\text{tension\_v} \times \text{courant\_a}}{1000}$$

2. **Le Rendement ($\%$)** du champ solaire de surface $A_{champ} = 50 \text{ m}^2$ :
   $$\text{rendement\_pct} = \frac{\text{puissance\_kw}}{\left(\frac{\text{irradiance\_w\_m2} \times 50}{1000}\right)} \times 100$$

3. **La catégorisation météo (`pd.cut`)** :
   On découpe l'irradiance en 3 niveaux d'ensoleillement :
   * `Faible` : entre 0 et 300 W/m²
   * `Moyen` : entre 300 et 700 W/m²
   * `Fort` : entre 700 et 1200 W/m²

---

### ✏️ Exercice (Quête 3)
Ajoute les colonnes `'puissance_kw'`, `'rendement_pct'` et `'regime_ensoleillement'` dans `df_valide`.

In [ ]:
# ==========================================
# QUÊTE 3 : Calculs de puissance, rendement et régimes météo
# ==========================================

surface_champ_m2 = 50.0  # Surface totale des panneaux en m²

# Consignes :
# 1. Dans `df_valide`, crée la colonne 'puissance_kw' : (tension_v * courant_a) / 1000.0
# 2. Crée la colonne 'rendement_pct' en appliquant la formule du cours ci-dessus.
# 3. Crée la colonne 'regime_ensoleillement' avec pd.cut() sur 'irradiance_w_m2' :
#    - bins = [0, 300, 700, 1200]
#    - labels = ['Faible', 'Moyen', 'Fort']

# ### START CODE HERE ###
df_valide['puissance_kw'] = (df_valide['tension_v'] * df_valide['courant_a']) / 1000.0

puissance_solaire_recue_kw = (df_valide['irradiance_w_m2'] * surface_champ_m2) / 1000.0
df_valide['rendement_pct'] = (df_valide['puissance_kw'] / puissance_solaire_recue_kw) * 100.0

df_valide['regime_ensoleillement'] = pd.cut(
    df_valide['irradiance_w_m2'],
    bins=[0, 300, 700, 1200],
    labels=['Faible', 'Moyen', 'Fort']
)
# ### END CODE HERE ###

# --- VERIFICATION AUTOMATIQUE ---
assert round(df_valide['rendement_pct'].mean(), 2) == 11.47, f"Rendement moyen attendu : 11.47 %, obtenu : {df_valide['rendement_pct'].mean():.2f}"
assert df_valide['regime_ensoleillement'].value_counts()['Moyen'] == 40, "Problème dans le comptage du régime 'Moyen'"
print("✅ Bravo ! Quête 3 réussie : Puissances, rendements et régimes météo correctement calculés.")

# --- AFFICHAGE DU GRAPHIQUE ---
from utils import tracer_rendement_physique
tracer_rendement_physique(df_valide)

## 🔴 Niveau 4 : Bilan Énergétique par Régime Météo (`groupby`)

### 📚 COURS : Regrouper des données physiques pour faire un bilan
Pour savoir combien d'énergie a été produite au total dans chaque condition météo, on utilise la commande `groupby()`. Elle permet de regrouper les lignes par catégorie (ex: `Faible`, `Moyen`, `Fort`) et de faire des calculs globaux (somme, moyenne).

Sintaxe à utiliser :
```python
df_kpi = df.groupby('regime_ensoleillement', observed=False).agg(
    puissance_totale_kw=('puissance_kw', 'sum'),
    rendement_moyen_pct=('rendement_pct', 'mean')
).reset_index()

In [ ]:
# ==========================================
# QUÊTE 4 : Synthèse des résultats par régime d'ensoleillement
# ==========================================

# Consignes :
# 1. Fais un groupby sur 'regime_ensoleillement' (avec `observed=False`).
# 2. Calcule :
#    - 'puissance_totale_kw' : la SOMME ('sum') de 'puissance_kw'
#    - 'rendement_moyen_pct' : la MOYENNE ('mean') de 'rendement_pct'
# 3. Réinitialise l'index avec .reset_index() et stocke dans `df_kpi_meteo`.

# ### START CODE HERE ###

# ### END CODE HERE ###

# --- VERIFICATION AUTOMATIQUE ---
assert len(df_kpi_meteo) == 3, "Le tableau doit contenir exactement 3 catégories"
puiss_fort = df_kpi_meteo.loc[df_kpi_meteo['regime_ensoleillement'] == 'Fort', 'puissance_totale_kw'].values[0]
assert round(puiss_fort, 1) == 91.6, f"Puissance attendue pour 'Fort' : 91.6 kW, obtenue : {puiss_fort:.1f}"
print("✅ Bravo ! Quête 4 réussie : Le bilan énergétique complet est généré.")

# --- AFFICHAGE DU GRAPHIQUE ---
from utils import tracer_kpi_regime
tracer_kpi_regime(df_kpi_meteo)

## 🟣 Niveau 5 : Visualisation Générale & Dashboard d'Audit

### 📚 COURS : Interpréter les graphiques pour un diagnostic physique
Un bon technicien ou ingénieur en énergie doit savoir lire des graphiques pour valider le fonctionnement de son installation :
* **Graphe 1 ($P_{él}$ vs $G$)** : La puissance électrique augmente de façon linéaire avec l'ensoleillement.
* **Graphe 2 (Tension $V$)** : La tension doit rester relativement stable aux alentours de 230V.
* **Graphe 3 (Rendement par Régime)** : Il permet d'observer la dispersion des performances.

---

### ✏️ Exercice (Quête 5)
Affiche le tableau de bord global en exécutant la fonction d'affichage fournie dans le module `utils.py`.

In [ ]:
# ==========================================
# QUÊTE 5 : Affichage du Dashboard Complet
# ==========================================

# Consigne :
# Exécute la fonction `tracer_dashboard_energetique_eda(df_valide)`.

# ### START CODE HERE ###
from utils import tracer_dashboard_energetique_eda

tracer_dashboard_energetique_eda(df_valide)
# ### END CODE HERE ###

print("✅ Bravo ! Quête 5 réussie : Dashboard affiché avec succès.")

## 👑 Bilan et Récapitulatif du TP (Niveau L3)

Bravo ! En complétant cet exercice, tu as acquis les compétences fondamentales de traitement de données pour les systèmes énergétiques :
1. **Compréhension des capteurs** : Manipulation des grandeurs $G, T, V, I$.
2. **Nettoyage de données (IQR)** : Élimination des valeurs aberrantes produites par des bugs matériels.
3. **Formulation physique** : Calcul vectorisé de la puissance électrique et du rendement du champ solaire.
4. **Discrétisation (`pd.cut`)** : Regroupement par régimes météo.
5. **Agrégation (`groupby`) & Diagnostic** : Calcul du bilan global de production d'énergie.

In [ ]:
from utils import afficher_bilan_projet_04

# Affiche le bilan final résumé dans la console
afficher_bilan_projet_04(df_pv, df_valide)